# OAKG AAAI 2027 — Experiment Driver

**Observability-Aware Knowledge Graph Reasoning for Heterogeneous Medical Imaging Data**

This notebook is a *thin driver*. All logic lives in the importable `oakg` package under
[`oakg/`](../oakg/) so every table and figure is reproducible without hidden notebook state.

- Demo mode (default) runs the full pipeline on a reproducible synthetic benchmark.
- Real mode: set `use_demo_data=False` and place the five benchmark CSVs in `data/`.

Deliverables are written under `results/` exactly as listed in Section 13 of the guidelines.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make `oakg` importable (package at repo root)

import pandas as pd
import oakg
from oakg import Config
from oakg.pipeline import run_all
print('oakg', oakg.__version__)


## 1. Configuration

Everything experiment-defining (seeds, thresholds, paths, bootstrap count) is fixed in `Config`
and archived to `results/config.json`. Switch to the real benchmark by editing this one object.


In [ ]:
ROOT = pathlib.Path.cwd().parent  # repository root (notebook lives in notebooks/)
config = Config(
    use_demo_data=True,     # set False and fill data/ with the five CSVs for the real study
    n_demo_cases=120,
    seed=2027,
    data_dir=ROOT / 'data',
    results_dir=ROOT / 'results_demo',  # demo only; real results/ untouched
    embeddings_dir=ROOT / 'embeddings',
)
config.to_dict()


## 2. Run the full pipeline

Regenerates every deliverable: query-level results, summaries, upstream degradation, ranking-policy
selection, risk–coverage curve + figure, ranking consistency, structured-query semantics, and the
publication-ready table with bootstrap CIs.


In [ ]:
out = run_all(config)
print('query-level rows:', out.results.shape[0])
print('methods x strata summarized:', out.summary.shape[0])


## 3. Retrieval summary (Tables A/B)


In [ ]:
(out.summary.sort_values('nDCG@10', ascending=False)
    [['track','masking_regime','method','P@10','R@10','nDCG@10','ServedRate']]
    .head(15))


## 4. Ranking-policy selection (Table C)

Primary policy chosen by validation nDCG@10 together with served-query rate. Confirm on the real
validation partition and freeze before test evaluation. Earlier results used `gamma*S` (product).


In [ ]:
out.policy_selection


## 5. Critical paper comparison — OAKG vs. masked cosine

Paired 95% bootstrap CI with Holm correction: does the graph-based observability mechanism add
value beyond pairwise feature masking?


In [ ]:
from oakg import comparison_family
subset = out.results[(out.results['track']=='ref') & (out.results['masking_regime']=='random')]
cmp = comparison_family(
    subset, 'OAKG-product [ref]',
    ['Masked cosine [ref]', 'Missingness indicators [ref]', 'Gower [ref]'],
    n_bootstrap=config.n_bootstrap, seed=config.seed,
)
cmp


## 6. Structured-query semantics (Table F)

Closed-world vs. generic open-world vs. OAKG support-aware three-valued (T/F/U) reasoning.


In [ ]:
pd.read_csv(config.tables_dir / 'structured_query_summary.csv')


## 7. Risk–coverage (Figure 1)


In [ ]:
from IPython.display import Image
Image(filename=str(config.figures_dir / 'risk_coverage_curve.png'))


## 8. Deliverables written

See Section 13 of the guidelines. Re-running this notebook regenerates all of them from `config`.


In [ ]:
for p in sorted(config.results_dir.rglob('*')):
    if p.is_file() and p.name != '.gitkeep':
        print(' -', p.relative_to(config.results_dir))
